<h2 style="text-align: center; background-color: #c1e7f6;color: black; font-family: serif; padding: 20px;">بِسْمِ ٱللَّٰهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ</h2>

<h3 style="text-align: center; background-color: #103456;color: white; font-family: serif; padding: 18px;">Welcome to Qasim's Academy</h3>

- My name is Muhammad Qasim Ashfaq, and I am a passionate software engineer and machine learning enthusiast.
- I have a strong background in computer science and a deep interest in artificial intelligence and data science.
- My goal is to share my knowledge and expertise with others through this academy, providing high-quality educational content and resources for learners of all levels.
- Whether you are a beginner looking to get started or an experienced professional seeking to enhance your skills, **Qasim's Academy** is here to support your learning journey.

# Decision Tree Classifier

A decision tree classifies data by asking a series of yes/no style questions, exactly like a flow chart.
In this notebook we will:

1. **Load** a tiny "should we play tennis today?" dataset
2. **Understand the maths** (entropy, information gain, Gini) and compute it by hand
3. **Build a decision tree from scratch** (the ID3 algorithm) in ~15 lines
4. **Preprocess** the categorical data properly
5. **Train and evaluate** the same model with scikit-learn
6. **Visualize** the tree and compare it with our from-scratch version
7. **Save** the model
8. **Build a web UI** with Gradio

> **Run it on Colab?** The next cell takes care of fetching the data for you.

In [1]:
# Setup: only does something when running on Google Colab
import os, sys, subprocess

if "google.colab" in sys.modules and not os.path.exists("data"):
    subprocess.run(["git", "clone", "-q", "https://github.com/notjustavibe/Machine-Learning.git"], check=True)
    os.chdir("Machine-Learning/Decision-Tree-Classifier")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)

os.makedirs("model", exist_ok=True)

# Step 1: Load the data

In [2]:
import pandas as pd

data = pd.read_csv("data/decisiontree.csv")   # same data as decisiontree.xlsx
data

,day,outlook,temperature,humidity,wind,decision
0,1,sunny,hot,high,weak,no
1,2,sunny,hot,high,strong,no
2,3,overcast,hot,high,weak,yes
3,4,rain,mild,high,weak,yes
4,5,rain,cool,normal,weak,yes
5,6,rain,cool,normal,strong,no
6,7,overcast,cool,normal,strong,yes
7,8,sunny,mild,high,weak,no
8,9,sunny,cool,normal,weak,yes
9,10,rain,mild,normal,weak,yes


Each row is one day. We want to predict **`decision`** (do we play tennis? yes/no) from the weather:

| Column | Meaning | Values |
|---|---|---|
| `day` | row number | 1 to 14 (just an ID, **not** a useful feature) |
| `outlook` | sky condition | sunny, overcast, rain |
| `temperature` | how warm | hot, mild, cool |
| `humidity` | how humid | high, normal |
| `wind` | wind strength | weak, strong |
| `decision` | **target** | yes, no |

In [3]:
data["decision"].value_counts()

decision
yes    9
no     5
Name: count, dtype: int64

# Step 2: The maths behind the tree

A tree has to choose **which question to ask first**. The best question is the one that splits the data into the *purest* groups, where a group is pure if it contains only "yes" or only "no".

## Entropy: how mixed is a group?

$$H(S) = -\sum_{i=1}^{c} p_i \log_2 p_i$$

- $S$ is the set of rows, $c$ the number of classes, $p_i$ the fraction of rows in class $i$
- Entropy is **0** for a perfectly pure group and **1** for a 50/50 mix (with two classes)

## Gini impurity: an alternative

$$Gini(S) = 1 - \sum_{i=1}^{c} p_i^2$$

Gini and entropy almost always produce very similar trees. Gini is slightly cheaper to compute (no logarithm), which is why it is scikit-learn's default.

## Information gain: how good is a question?

$$IG(S, A) = H(S) - \sum_{v \in A} \frac{|S_v|}{|S|} H(S_v)$$

It is the entropy *before* the split minus the weighted-average entropy *after* it. **The feature with the highest gain wins.**

In [4]:
import numpy as np

def entropy(labels):
    p = labels.value_counts(normalize=True)
    return -(p * np.log2(p)).sum() + 0.0   # + 0.0 avoids printing -0.000

def gini(labels):
    p = labels.value_counts(normalize=True)
    return 1 - (p ** 2).sum()

print(f"Entropy of the whole dataset (9 yes / 5 no): {entropy(data['decision']):.3f}")
print(f"Gini    of the whole dataset              : {gini(data['decision']):.3f}")

Entropy of the whole dataset (9 yes / 5 no): 0.940
Gini    of the whole dataset              : 0.459


`scipy` can do the same thing in one line, which is a nice way to double-check our function:

In [5]:
from scipy.stats import entropy as scipy_entropy

scipy_entropy([9, 5], base=2)   # counts of yes / no

np.float64(0.940285958670631)

### Worked example: splitting on `outlook`

Let's look at how the outcome is distributed inside each outlook value:

In [6]:
pd.crosstab(data["outlook"], data["decision"])

decision,no,yes
outlook,,
overcast,0,4
rain,2,3
sunny,3,2


In [7]:
for value, subset in data.groupby("outlook"):
    print(f"outlook = {value:9s} -> {len(subset):2d} rows, entropy = {entropy(subset['decision']):.3f}")

outlook = overcast  ->  4 rows, entropy = 0.000
outlook = rain      ->  5 rows, entropy = 0.971
outlook = sunny     ->  5 rows, entropy = 0.971


`overcast` is perfectly pure (4 yes, 0 no), so its entropy is 0. Now the information gain of **every** feature:

In [8]:
def information_gain(df, feature, target="decision"):
    before = entropy(df[target])
    after = sum(len(sub) / len(df) * entropy(sub[target]) for _, sub in df.groupby(feature))
    return before - after

features = ["outlook", "temperature", "humidity", "wind"]
gains = pd.Series({f: information_gain(data, f) for f in features}).sort_values(ascending=False)
gains.round(3)

outlook        0.247
humidity       0.152
wind           0.048
temperature    0.029
dtype: float64

`outlook` has the highest information gain, so it becomes the **root** of the tree.

# Step 3: A decision tree from scratch (ID3)

The ID3 algorithm is just this loop:

1. If all rows have the same label, stop and make a **leaf**
2. If there are no features left, stop and predict the majority label
3. Otherwise pick the feature with the highest information gain, split on it, and **repeat for each branch**

In [9]:
def id3(df, features, target="decision", depth=0):
    indent = "    " * depth
    labels = df[target]

    if labels.nunique() == 1:                 # pure -> leaf
        print(f"{indent}-> {labels.iloc[0]}")
        return
    if not features:                          # nothing left to split on -> majority vote
        print(f"{indent}-> {labels.mode()[0]}")
        return

    best = max(features, key=lambda f: information_gain(df, f, target))
    remaining = [f for f in features if f != best]
    for value, subset in df.groupby(best):
        print(f"{indent}{best} = {value}")
        id3(subset, remaining, target, depth + 1)

id3(data, features)

outlook = overcast
    -> yes
outlook = rain
    wind = strong
        -> no
    wind = weak
        -> yes
outlook = sunny
    humidity = high
        -> no
    humidity = normal
        -> yes


Read it like a flow chart: if the outlook is `overcast` we always play; if it is `rain` we check the wind; if it is `sunny` we check the humidity.

# Step 4: Preprocessing for scikit-learn

scikit-learn trees need numbers, not text. There are two common ways to convert categories:

| Method | Example | Catch |
|---|---|---|
| `LabelEncoder` / `OrdinalEncoder` | sunny=2, rain=1, overcast=0 | Invents an *order* that does not exist, and `LabelEncoder` is meant for the **target**, not the features |
| **`OneHotEncoder`** | `outlook_sunny` = 1/0 | One extra column per value, but no fake order, and the tree becomes easy to read |

We use **one-hot encoding**, and we put it **inside a `Pipeline`**. That way the encoding rules are saved together with the model, so we can feed raw text like `"sunny"` at prediction time and never have to copy mappings around by hand.

In [10]:
X = data.drop(columns=["day", "decision"])   # "day" is only an ID, so we drop it
y = data["decision"]
X.head()

,outlook,temperature,humidity,wind
0,sunny,hot,high,weak
1,sunny,hot,high,strong
2,overcast,hot,high,weak
3,rain,mild,high,weak
4,rain,cool,normal,weak


# Step 5: Train/test split and training

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y   # stratify keeps the yes/no ratio similar
)

def make_model(**tree_params):
    preprocess = ColumnTransformer([("onehot", OneHotEncoder(handle_unknown="ignore"), list(X.columns))])
    tree = DecisionTreeClassifier(criterion="entropy", random_state=42, **tree_params)
    return Pipeline([("preprocess", preprocess), ("tree", tree)])

model = make_model()
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('tree', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['no','yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['outlook','temperature','humidity','wind']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not speci

# Step 6: Evaluation

## Test-set accuracy

In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)
print(f"Test samples: {len(X_test)}")
print(f"Accuracy    : {accuracy_score(y_test, y_pred):.2f}")
confusion_matrix(y_test, y_pred, labels=["no", "yes"])

Test samples: 4
Accuracy    : 0.75


array([[0, 1],
       [0, 3]])

**Be careful with this number.** With only 14 rows our test set has just 4 of them, so one single mistake changes the accuracy by 25 points. A score from 4 samples tells us almost nothing, so let's use a more honest method.

## Cross-validation

Cross-validation splits the data into several folds and uses each fold as the test set once, so *every* row gets tested.

In [13]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.dummy import DummyClassifier

scores = cross_val_score(make_model(), X, y, cv=cv)
baseline = cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=cv)
print("Accuracy per fold   :", scores.round(2))
print(f"Tree, mean accuracy : {scores.mean():.2f} (+/- {scores.std():.2f})")
print(f"Baseline (always say 'yes'): {baseline.mean():.2f}")

Accuracy per fold   : [0.67 0.33 0.33 0.33 0.5 ]
Tree, mean accuracy : 0.43 (+/- 0.13)
Baseline (always say 'yes'): 0.63


**Reading the result honestly:** on 14 rows the tree does *not* clearly beat the "always say yes" baseline. That is normal. Each fold trains on only ~11 rows and cannot learn all the branches. This dataset is a teaching toy (the classic "play tennis" example), not something to judge model quality on. What matters here is *how* the algorithm works, and the habit of checking against a baseline.

## Overfitting and `max_depth`

An unrestricted tree keeps splitting until every leaf is pure, which means it can **memorize** the training data. Limiting the depth is the simplest way to fight that. Let's compare training accuracy with cross-validated accuracy:

In [14]:
rows = []
for depth in [1, 2, 3, 4, None]:
    m = make_model(max_depth=depth)
    train_acc = m.fit(X, y).score(X, y)
    cv_acc = cross_val_score(make_model(max_depth=depth), X, y, cv=cv).mean()
    rows.append({"max_depth": str(depth), "train accuracy": round(train_acc, 2), "cross-val accuracy": round(cv_acc, 2)})

pd.DataFrame(rows)

,max_depth,train accuracy,cross-val accuracy
0,1,0.64,0.37
1,2,0.86,0.43
2,3,0.86,0.37
3,4,1.00,0.43
4,None,1.00,0.43


If the training accuracy is much higher than the cross-validated accuracy, the tree is memorizing. With real, larger datasets you would tune `max_depth`, `min_samples_leaf` and friends with `GridSearchCV`.

# Step 7: Visualize the tree

For the final model we train on **all** the data, since there is nothing left to "test" on a 14-row dataset and a deployed model should use every example available.

In [15]:
final_model = make_model().fit(X, y)

tree = final_model.named_steps["tree"]
feature_names = [n.split("__", 1)[1] for n in final_model.named_steps["preprocess"].get_feature_names_out()]

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree, export_text

plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=feature_names, class_names=list(tree.classes_), filled=True, rounded=True, fontsize=9)
plt.show()

In [16]:
print(export_text(tree, feature_names=feature_names))

|--- outlook_overcast <= 0.50
|   |--- humidity_normal <= 0.50
|   |   |--- outlook_rain <= 0.50
|   |   |   |--- class: no
|   |   |--- outlook_rain >  0.50
|   |   |   |--- wind_strong <= 0.50
|   |   |   |   |--- class: yes
|   |   |   |--- wind_strong >  0.50
|   |   |   |   |--- class: no
|   |--- humidity_normal >  0.50
|   |   |--- wind_strong <= 0.50
|   |   |   |--- class: yes
|   |   |--- wind_strong >  0.50
|   |   |   |--- outlook_sunny <= 0.50
|   |   |   |   |--- class: no
|   |   |   |--- outlook_sunny >  0.50
|   |   |   |   |--- class: yes
|--- outlook_overcast >  0.50
|   |--- class: yes



Compare it with our from-scratch tree. Both **start with `outlook`**, and both isolate `overcast` as an all-"yes" branch. The shapes differ because scikit-learn implements **CART**, which only makes *binary* splits (a condition like `outlook_overcast <= 0.5` means "outlook is *not* overcast"), while ID3 splits into one branch per value. Same idea, slightly different bookkeeping.

## Which features mattered most?

In [17]:
importances = pd.Series(tree.feature_importances_, index=feature_names).sort_values()
importances[importances > 0].plot.barh(title="Feature importance", figsize=(7, 4))
plt.show()

# Step 8: Save the model

We save the **whole pipeline** (encoder + tree) with `joblib`, which is the standard tool for scikit-learn models.

> **Good to know:** a saved model should be loaded with the same scikit-learn version that created it, and you should **never load model files from sources you do not trust** (pickle-based files can run code when loaded).

In [18]:
import joblib, sklearn

joblib.dump(final_model, "model/decision_tree_model.joblib")
print("Saved with scikit-learn", sklearn.__version__)

Saved with scikit-learn 1.9.1


## Load it back and predict

In [19]:
loaded_model = joblib.load("model/decision_tree_model.joblib")

new_day = pd.DataFrame([{"outlook": "sunny", "temperature": "mild", "humidity": "high", "wind": "weak"}])
print("Prediction   :", loaded_model.predict(new_day)[0])
print("Probabilities:", {c: round(float(p), 2) for c, p in zip(loaded_model.classes_, loaded_model.predict_proba(new_day)[0])})

Prediction   : no
Probabilities: {'no': 1.0, 'yes': 0.0}


Because the encoder lives inside the pipeline, we pass plain text. No hand-written mappings, so no way for the UI and the model to disagree.

# Step 9: A web UI with Gradio

[Gradio](https://www.gradio.app) turns a Python function into a web app. We need three things: a **function**, its **inputs** and its **outputs**.

In [20]:
import gradio as gr

CHOICES = {col: sorted(data[col].unique()) for col in features}

def predict_play(outlook, temperature, humidity, wind):
    row = pd.DataFrame([{"outlook": outlook, "temperature": temperature, "humidity": humidity, "wind": wind}])
    proba = dict(zip(loaded_model.classes_, loaded_model.predict_proba(row)[0]))
    return {"Play (yes)": float(proba["yes"]), "Don't play (no)": float(proba["no"])}

demo = gr.Interface(
    fn=predict_play,
    inputs=[gr.Dropdown(CHOICES[col], value=CHOICES[col][0], label=col.capitalize()) for col in features],
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    examples=[["sunny", "hot", "high", "weak"], ["overcast", "cool", "normal", "strong"], ["rain", "mild", "high", "strong"]],
    title="Decision Tree Classifier: play tennis?",
    description="Pick the weather conditions and the model predicts whether to play.",
    flagging_mode="never",
)

demo.launch()

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


The same app is available as a standalone script, `app.py`, in this folder. Run `python app.py` to launch it outside the notebook, or deploy it for free on [Hugging Face Spaces](https://huggingface.co/spaces).

# Summary

- A decision tree repeatedly picks the question with the highest **information gain** (or lowest Gini impurity).
- We built ID3 from scratch. scikit-learn (CART) chooses the same first question, `outlook`, but with binary splits.
- Tiny datasets need **cross-validation**, and a single test score from 4 rows is not reliable.
- Putting preprocessing **inside a Pipeline** makes saving, loading and building a UI much safer.

## Exercises

1. Change `criterion="entropy"` to `"gini"`. Does the tree change?
2. Add `max_depth=2` to the final model. How does the tree and its cross-validated accuracy change?
3. Add a new row to the CSV that contradicts an existing one. What happens to the tree?
4. Add a `gr.Slider` or `gr.Radio` input in place of a dropdown.

---
### Regards, Muhammad Qasim Ashfaq